# 03 — Inference

Loads a fold checkpoint trained by `02_train.ipynb` / `src/train.py` and runs it against the real competition test set, mirroring `src/infer.py`.

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd, torch
from transformers import AutoTokenizer
from src.modeling import DebertaRegressor
from src.train import load_config

cfg = load_config('../configs/default.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
test = pd.read_csv('../data/test.csv')
test.head()

In [ ]:
model_name = cfg.get('model_dir') or cfg['model_name']
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = DebertaRegressor(model_name, num_labels=cfg['num_labels']).to(device)
# model.load_state_dict(torch.load('../outputs/fold0/model.pt')['model_state'])
model.eval()
print('model ready')

In [ ]:
enc = tokenizer(list(test['full_text']), truncation=True, padding='max_length',
                 max_length=cfg['max_length'], return_tensors='pt')
with torch.no_grad():
    out = model(input_ids=enc['input_ids'].to(device),
                attention_mask=enc['attention_mask'].to(device))
preds = out['logits'].squeeze(-1).cpu().numpy()
print(preds)

In [ ]:
from src.qwk import apply_thresholds
import json
# thresholds saved from the real training run (see reports/experiments_summary.json)
sub = test[['essay_id']].copy()
sub['score'] = preds.clip(1, 6).round().astype(int)
sub.to_csv('../artifacts_submission_demo.csv', index=False)
sub.head()